In [3]:
import duckdb

In [1]:
import pandas as pd

customers = pd.read_csv("/Users/shash/Documents/Data_Science/EDA/Data/customers.csv")
order_items = pd.read_csv("/Users/shash/Documents/Data_Science/EDA/Data/order_items.csv")
orders = pd.read_csv("/Users/shash/Documents/Data_Science/EDA/Data/orders.csv")
products = pd.read_csv("/Users/shash/Documents/Data_Science/EDA/Data/products.csv")
sellers = pd.read_csv("/Users/shash/Documents/Data_Science/EDA/Data/sellers.csv")
shipments = pd.read_csv("/Users/shash/Documents/Data_Science/EDA/Data/shipments.csv")

## Data Quality

In [66]:
# checking duplicates as one table
duckdb.sql("SELECT COUNT(*) AS total_rows, COUNT(DISTINCT shipment_id) AS unique_ids FROM shipments")

┌────────────┬────────────┐
│ total_rows │ unique_ids │
│   int64    │   int64    │
├────────────┼────────────┤
│      91994 │      91994 │
└────────────┴────────────┘

In [53]:
# Data quality check
# to check on Primary key uniqueness
primary_key = {'customers':'customer_id', 'sellers':'seller_id', 
            'products': 'product_id', 'orders':'order_id',
            'order_items':'order_item_id', 'shipments':'shipment_id'}

for table, pk in primary_key.items():
    result = duckdb.sql(f"""
        SELECT 
            COUNT(*) AS total_rows, 
            COUNT(DISTINCT {pk}) AS unique_ids 
        FROM {table}
    """).fetchone()

total_rows, unique_ids = result
print(result)

(91994, 91994)


### Check for null values

In [54]:
# all datasets as one dict
dfs = {
    'customers': customers,
    'sellers': sellers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
    'shipments': shipments
}

In [55]:
for name, df in dfs.items():
    nulls = df.isnull().sum().sum()
    if nulls > 0:
        print(f"{name}: {nulls} missing values")
    else:
        print(f"{name}: No nulls presents")

customers: No nulls presents
sellers: No nulls presents
products: No nulls presents
orders: No nulls presents
order_items: No nulls presents
shipments: 5473 missing values


In [56]:
duckdb.sql("select count(*) from shipments where delivery_status in ('Lost', 'InTransit')")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         5473 │
└──────────────┘

### Check for Date consistency
Hypothesis: Delivered Date >= Shipping Date >= Ordered Date

In [61]:
duckdb.sql(" select s.shipped_at, s.delivered_at, o.created_at from shipments s join orders o on o.order_id = s.order_id  where delivery_status not in ('Lost','InTransit')")

┌─────────────────────┬─────────────────────┬────────────┐
│     shipped_at      │    delivered_at     │ created_at │
│       varchar       │       varchar       │  varchar   │
├─────────────────────┼─────────────────────┼────────────┤
│ 2024-07-01 13:00:27 │ 2024-07-02 18:41:52 │ 2024-07-01 │
│ 2024-07-01 16:17:32 │ 2024-07-03 04:35:14 │ 2024-07-01 │
│ 2024-07-01 14:39:14 │ 2024-07-02 21:48:54 │ 2024-07-01 │
│ 2024-07-01 15:26:20 │ 2024-07-02 22:42:59 │ 2024-07-01 │
│ 2024-07-01 16:03:13 │ 2024-07-02 05:22:51 │ 2024-07-01 │
│ 2024-07-01 12:19:03 │ 2024-07-02 23:15:02 │ 2024-07-01 │
│ 2024-07-01 04:52:37 │ 2024-07-02 00:10:45 │ 2024-07-01 │
│ 2024-07-01 15:29:58 │ 2024-07-02 07:33:31 │ 2024-07-01 │
│ 2024-07-01 15:54:45 │ 2024-07-02 16:31:20 │ 2024-07-01 │
│ 2024-07-01 10:53:41 │ 2024-07-07 17:06:34 │ 2024-07-01 │
│          ·          │          ·          │     ·      │
│          ·          │          ·          │     ·      │
│          ·          │          ·          │     ·     

In [75]:
query = f"""
select 
    count(*),
    sum (case when created_at < shipped_at then 0 else 1 end) as shipped_before_order,
    sum (case when shipped_at < delivered_at then 0 else 1 end) as deliver_before_shipped,
    sum (case when created_at < delivered_at then 0 else 1 end) as delivered_before_ordered
from orders o
join shipments s on o.order_id = s.order_id
where s.delivery_status not in ('Lost', 'InTransit')
"""

In [76]:
duckdb.sql(query)

┌──────────────┬──────────────────────┬────────────────────────┬──────────────────────────┐
│ count_star() │ shipped_before_order │ deliver_before_shipped │ delivered_before_ordered │
│    int64     │        int128        │         int128         │          int128          │
├──────────────┼──────────────────────┼────────────────────────┼──────────────────────────┤
│        86521 │                    0 │                      0 │                        0 │
└──────────────┴──────────────────────┴────────────────────────┴──────────────────────────┘

## Tasks and Deliverables

### 1a. Monthly GMV by City and Category
Consideration: I have assumed the Customer's City as the field.

In [ ]:
gmv_query = f"""
select 
    date_trunc('month', o.created_at::TIMESTAMP) as order_month, c.city as City,
    p.category as Category, sum(quantity*unit_price) as GMV from order_items oi
    join products p on p.product_id = oi.product_id
    join orders o on o.order_id=oi.order_id
join customers c on c.customer_id = o.customer_id 
group by order_month, category, city
order by order_month desc, gmv desc"""

In [154]:
duckdb.sql(gmv_query)

┌─────────────────────┬────────────┬────────────────┬────────────┐
│     order_month     │    City    │    Category    │    GMV     │
│      timestamp      │  varchar   │    varchar     │   double   │
├─────────────────────┼────────────┼────────────────┼────────────┤
│ 2025-12-01 00:00:00 │ Mumbai     │ Electronics    │ 26230729.0 │
│ 2025-12-01 00:00:00 │ Delhi      │ Electronics    │ 21437112.0 │
│ 2025-12-01 00:00:00 │ Bangalore  │ Electronics    │ 18733801.0 │
│ 2025-12-01 00:00:00 │ Hyderabad  │ Electronics    │ 14339461.0 │
│ 2025-12-01 00:00:00 │ Pune       │ Electronics    │ 12332499.0 │
│ 2025-12-01 00:00:00 │ Chennai    │ Electronics    │ 10186456.0 │
│ 2025-12-01 00:00:00 │ Kolkata    │ Electronics    │  9144426.0 │
│ 2025-12-01 00:00:00 │ Ahmedabad  │ Electronics    │  7078199.0 │
│ 2025-12-01 00:00:00 │ Jaipur     │ Electronics    │  5837281.0 │
│ 2025-12-01 00:00:00 │ Mumbai     │ Home & Kitchen │  5075594.0 │
│          ·          │   ·        │    ·           │       · 

### 1b. Monthly count of orders and Unique active customers

In [ ]:
uniq_query = f"""
select date_trunc('month', o.created_at::TIMESTAMP) as Order_month,
    count(distinct o.order_id) as Monthly_Orders, count(distinct c.customer_id) as Unique_Customers 
    from orders o
join customers c on c.customer_id = o.customer_id 
group by order_month
order by order_month desc"""

In [196]:
duckdb.sql(uniq_query)

┌─────────────────────┬────────────────┬──────────────────┐
│     Order_month     │ Monthly_Orders │ Unique_Customers │
│      timestamp      │     int64      │      int64       │
├─────────────────────┼────────────────┼──────────────────┤
│ 2025-12-01 00:00:00 │           5466 │             4836 │
│ 2025-11-01 00:00:00 │           5491 │             4876 │
│ 2025-10-01 00:00:00 │           5722 │             5006 │
│ 2025-09-01 00:00:00 │           5380 │             4743 │
│ 2025-08-01 00:00:00 │           5659 │             4950 │
│ 2025-07-01 00:00:00 │           5580 │             4912 │
│ 2025-06-01 00:00:00 │           5463 │             4838 │
│ 2025-05-01 00:00:00 │           5536 │             4850 │
│ 2025-04-01 00:00:00 │           5515 │             4866 │
│ 2025-03-01 00:00:00 │           5627 │             4964 │
│ 2025-02-01 00:00:00 │           5163 │             4581 │
│ 2025-01-01 00:00:00 │           5604 │             4898 │
│ 2024-12-01 00:00:00 │           5719 │

### B1. Monthly Marketplace metrics: Orders, Unique customers, Repeat purchase Rate
Definition considered: RPR = 100 x ((number of repeat customers)/(total number of unique customers))

In [270]:
b1query = f"""
with repeat_criteria as 
(select customer_id, min(created_at::timestamp)::date as repeat_at
    from (select customer_id, created_at,
    row_number() over (partition by customer_id order by created_at) as rn
    from orders 
where status = 'Delivered')
where rn = 2 
group by customer_id)

select 
    date_trunc('month', o.created_at::timestamp)::date as month,c.city,
    sum(oi.quantity * oi.unit_price) as gmv,
    count(distinct o.order_id) as number_of_orders,
    count(distinct o.customer_id) as unique_customers,
    round(count(distinct case when rc.repeat_at <= last_day(o.created_at::timestamp) then o.customer_id 
    end) * 100 / count(distinct o.customer_id),1) as repeat_purchase_rate
from orders o
join customers c on o.customer_id = c.customer_id
join order_items oi on o.order_id = oi.order_id
left join repeat_criteria rc on o.customer_id = rc.customer_id
group by month, c.city
order by month desc, gmv desc;
"""

In [271]:
duckdb.sql(b1query)

┌────────────┬────────────┬────────────┬──────────────────┬──────────────────┬──────────────────────┐
│   month    │    city    │    gmv     │ number_of_orders │ unique_customers │ repeat_purchase_rate │
│    date    │  varchar   │   double   │      int64       │      int64       │        double        │
├────────────┼────────────┼────────────┼──────────────────┼──────────────────┼──────────────────────┤
│ 2025-12-01 │ Mumbai     │ 34234584.0 │              959 │              865 │                 92.3 │
│ 2025-12-01 │ Delhi      │ 28732180.0 │              880 │              773 │                 91.2 │
│ 2025-12-01 │ Bangalore  │ 24793133.0 │              732 │              653 │                 91.7 │
│ 2025-12-01 │ Hyderabad  │ 19054348.0 │              594 │              510 │                 91.4 │
│ 2025-12-01 │ Pune       │ 16027369.0 │              476 │              420 │                 91.0 │
│ 2025-12-01 │ Chennai    │ 13913470.0 │              486 │              434 │    

### B2. First order delayed vs Ontime Repeat rate

In [ ]:
b2query_rank_order = f"""
with ranked_order as
    (select o.customer_id, o.created_at::timestamp as order_date,
    s.delivery_status, row_number() over (partition by o.customer_id order by o.created_at) as rn from orders o
join shipments s on s.order_id = o.order_id 
where o.status = 'Delivered'),

first_order as
    (select customer_id, order_date as first_date,
    case when lower(delivery_status) = 'ontime' then 'ontime' else 'delayed' 
    end as first_order_delay_status from ranked_order
where rn = 1),

second_order as
(select customer_id,order_date as second_date
from ranked_order
where rn = 2)

select f.first_order_delay_status,
round(count(s.customer_id) * 1.0 / count(f.customer_id), 4) as ninety_day_repeat_rate
from first_order f
left join second_order s 
on f.customer_id = s.customer_id and date_diff('day', f.first_date, s.second_date) <= 90
group by f.first_order_delay_status;
"""



In [217]:
duckdb.sql(b2query_rank_order)

┌──────────────────────────┬────────────────────────┐
│ first_order_delay_status │ ninety_day_repeat_rate │
│         varchar          │         double         │
├──────────────────────────┼────────────────────────┤
│ delayed                  │                 0.3948 │
│ ontime                   │                 0.4059 │
└──────────────────────────┴────────────────────────┘

### B3. Seller-Carrier Performance
THere is no seller, carrier, city combination with more than 100 deliveries

In [338]:
b3 = f"""
select
    oi.seller_id,
    s.carrier,
    s.ship_to_city,
    sum(oi.quantity * oi.unit_price) as total_gmv,
    sum(case when s.delivery_status <> 'OnTime' then oi.quantity * oi.unit_price else 0 end) as delayed_gmv,
    round(count(case when s.delivery_status <> 'OnTime' then s.order_id end)/ count(s.order_id), 4) as delayed_order_rate,
    round(avg(case when s.delivery_status <> 'OnTime' 
    then datediff('day', o.promised_delivery_date::timestamp, s.delivered_at::timestamp) 
    end), 2) as avg_delay_days
from shipments s
join orders o on o.order_id = s.order_id
join order_items oi on oi.order_id = s.order_id
where s.delivery_status in ('OnTime', 'Late_1_2d', 'Late_3_5d', 'Late_5p')
and o.status = 'Delivered'
group by oi.seller_id, s.carrier, s.ship_to_city
having count(s.order_id) >= 100
order by delayed_order_rate desc;
"""

In [337]:
duckdb.sql(b3)

┌───────────┬─────────┬──────────────┬───────────┬─────────────┬────────────────────┬────────────────┐
│ seller_id │ carrier │ ship_to_city │ total_gmv │ delayed_gmv │ delayed_order_rate │ avg_delay_days │
│  varchar  │ varchar │   varchar    │  double   │   double    │       double       │     double     │
├───────────┼─────────┼──────────────┼───────────┼─────────────┼────────────────────┼────────────────┤
│ S0109     │ InHouse │ Mumbai       │  570175.0 │      9103.0 │             0.1429 │            0.0 │
│ S0195     │ InHouse │ Mumbai       │  696794.0 │         0.0 │                0.0 │           NULL │
└───────────┴─────────┴──────────────┴───────────┴─────────────┴────────────────────┴────────────────┘

### B4. Query optimize

Understanding: The provided code first looks into the Shipment for the Delivered_at field and then again scans for the delivery_status to exclude the OnTIme deliveries.  

In my revised version, I have filtered down only the Delayed shipments and so the lookup happens only once.

In [221]:
# Provided query
b4 = f"""
SELECT o.order_id,
    (SELECT delivered_at
FROM shipments s
WHERE s.order_id = o.order_id) AS delivered_at,
o.created_at,
c.city
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
WHERE EXISTS (
SELECT 1
FROM shipments s2

WHERE s2.order_id = o.order_id
AND s2.delivery_status <> 'OnTime'
);
"""

In [222]:
duckdb.sql(b4)

┌───────────┬─────────────────────┬────────────┬───────────┐
│ order_id  │    delivered_at     │ created_at │   city    │
│  varchar  │       varchar       │  varchar   │  varchar  │
├───────────┼─────────────────────┼────────────┼───────────┤
│ ORD078367 │ 2025-09-05 03:18:26 │ 2025-09-03 │ Delhi     │
│ ORD078371 │ 2025-09-05 12:08:08 │ 2025-09-03 │ Chennai   │
│ ORD078376 │ 2025-09-05 16:43:09 │ 2025-09-03 │ Mumbai    │
│ ORD078384 │ 2025-09-06 14:45:34 │ 2025-09-03 │ Chennai   │
│ ORD078396 │ NULL                │ 2025-09-03 │ Pune      │
│ ORD078401 │ NULL                │ 2025-09-03 │ Mumbai    │
│ ORD078411 │ 2025-09-05 00:55:45 │ 2025-09-03 │ Kolkata   │
│ ORD078412 │ 2025-09-05 00:32:33 │ 2025-09-03 │ Mumbai    │
│ ORD078413 │ NULL                │ 2025-09-03 │ Chennai   │
│ ORD078414 │ 2025-09-06 08:59:31 │ 2025-09-03 │ Kolkata   │
│     ·     │  ·                  │     ·      │    ·      │
│     ·     │  ·                  │     ·      │    ·      │
│     ·     │  ·        

In [223]:
# revised query

b4_revised = f"""
with delayed_data as
(select order_id, delivered_at
from shipments
where lower(delivery_status) <> 'ontime'
)

select o.order_id, d.delivered_at, o.created_at, c.city
from orders o
join delayed_data d on o.order_id = d.order_id
join customers c on o.customer_id = c.customer_id;
"""

In [224]:
duckdb.sql(b4_revised)

┌───────────┬─────────────────────┬────────────┬────────────┐
│ order_id  │    delivered_at     │ created_at │    city    │
│  varchar  │       varchar       │  varchar   │  varchar   │
├───────────┼─────────────────────┼────────────┼────────────┤
│ ORD000002 │ 2024-07-03 04:35:14 │ 2024-07-01 │ Kochi      │
│ ORD000012 │ 2024-07-07 17:06:34 │ 2024-07-01 │ Lucknow    │
│ ORD000016 │ 2024-07-03 02:04:06 │ 2024-07-01 │ Chennai    │
│ ORD000018 │ 2024-07-03 03:56:20 │ 2024-07-01 │ Pune       │
│ ORD000022 │ NULL                │ 2024-07-01 │ Hyderabad  │
│ ORD000030 │ 2024-07-04 07:34:28 │ 2024-07-01 │ Mumbai     │
│ ORD000031 │ NULL                │ 2024-07-01 │ Hyderabad  │
│ ORD000045 │ NULL                │ 2024-07-01 │ Kolkata    │
│ ORD000046 │ NULL                │ 2024-07-01 │ Mumbai     │
│ ORD000056 │ 2024-07-03 23:14:37 │ 2024-07-01 │ Hyderabad  │
│     ·     │  ·                  │     ·      │   ·        │
│     ·     │  ·                  │     ·      │   ·        │
│     · 

### EDA to get insights

In [268]:
queryed = f"""
select delivery_status, carrier,  count(*) as deliveries,
sum (count(*)) over (partition by carrier) as total_carrier_deliveries, 
round(count(*)*100/ sum(count(*)) over(partition by carrier), 1) as percent
from shipments
group by carrier, delivery_status
order by delivery_status, carrier;"""

In [269]:
duckdb.sql(queryed)

┌─────────────────┬───────────┬────────────┬──────────────────────────┬─────────┐
│ delivery_status │  carrier  │ deliveries │ total_carrier_deliveries │ percent │
│     varchar     │  varchar  │   int64    │          int128          │ double  │
├─────────────────┼───────────┼────────────┼──────────────────────────┼─────────┤
│ InTransit       │ BlueDart  │       1165 │                    21906 │     5.3 │
│ InTransit       │ Delhivery │       1621 │                    29539 │     5.5 │
│ InTransit       │ Ekart     │       1170 │                    21840 │     5.4 │
│ InTransit       │ InHouse   │       1087 │                    18709 │     5.8 │
│ Late_1_2d       │ BlueDart  │       5522 │                    21906 │    25.2 │
│ Late_1_2d       │ Delhivery │       7701 │                    29539 │    26.1 │
│ Late_1_2d       │ Ekart     │       5866 │                    21840 │    26.9 │
│ Late_1_2d       │ InHouse   │       1299 │                    18709 │     6.9 │
│ Late_3_5d     

In [276]:
sellerq = f"""
with seller_performance as
(select oi.seller_id,
count(distinct o.order_id) as total_delivered_orders,
count(distinct case when lower(s.delivery_status) <> 'ontime' then o.order_id 
end) as delayed_orders from orders o
join order_items oi on o.order_id = oi.order_id
join shipments s on o.order_id = s.order_id
where lower(o.status) = 'delivered'
group by oi.seller_id
)

select seller_id, total_delivered_orders, delayed_orders,
round(delayed_orders * 100.0 / total_delivered_orders, 1) as delay_rate
from seller_performance
where total_delivered_orders >= 10
order by delay_rate desc;
"""

In [277]:
duckdb.sql(sellerq)

┌───────────┬────────────────────────┬────────────────┬────────────┐
│ seller_id │ total_delivered_orders │ delayed_orders │ delay_rate │
│  varchar  │         int64          │     int64      │   double   │
├───────────┼────────────────────────┼────────────────┼────────────┤
│ S0006     │                    349 │            183 │       52.4 │
│ S0019     │                    387 │            199 │       51.4 │
│ S0029     │                    327 │            166 │       50.8 │
│ S0017     │                    371 │            187 │       50.4 │
│ S0013     │                    378 │            189 │       50.0 │
│ S0033     │                    356 │            178 │       50.0 │
│ S0009     │                    342 │            171 │       50.0 │
│ S0038     │                    351 │            175 │       49.9 │
│ S0001     │                    326 │            162 │       49.7 │
│ S0005     │                    339 │            168 │       49.6 │
│   ·       │                     

In [294]:
lane_data = f""" 
with friction_data as (
    select 
        sl.primary_city as origin,
        c.city as destination,
        s.carrier,
        oi.seller_id,
        count(o.order_id) as total_orders,
        -- GMV tied to delayed orders
        sum(case when lower(s.delivery_status) <> 'ontime' then oi.quantity * oi.unit_price else 0 end) as delayed_gmv,
        -- Delay Rate
        count(case when lower(s.delivery_status) <> 'ontime' then 1 end) * 100.0 / count(*) as delay_rate
    from orders o
    join order_items oi on o.order_id = oi.order_id
    join sellers sl on oi.seller_id = sl.seller_id
    join customers c on o.customer_id = c.customer_id
    join shipments s on o.order_id = s.order_id
    where lower(o.status) = 'delivered'
    group by 1, 2, 3, 4
)
select * from friction_data
where total_orders >= 10
order by delayed_gmv desc, origin, destination, delayed_gmv desc
limit 30;
"""

In [295]:
duckdb.sql(lane_data)

┌────────────┬─────────────┬───────────┬───────────┬──────────────┬─────────────┬────────────────────┐
│   origin   │ destination │  carrier  │ seller_id │ total_orders │ delayed_gmv │     delay_rate     │
│  varchar   │   varchar   │  varchar  │  varchar  │    int64     │   double    │       double       │
├────────────┼─────────────┼───────────┼───────────┼──────────────┼─────────────┼────────────────────┤
│ Pune       │ Mumbai      │ Delhivery │ S0371     │           29 │    606170.0 │ 44.827586206896555 │
│ Chandigarh │ Delhi       │ Delhivery │ S0232     │           17 │    591036.0 │  47.05882352941177 │
│ Kolkata    │ Lucknow     │ Delhivery │ S0165     │           11 │    561482.0 │              100.0 │
│ Bangalore  │ Mumbai      │ BlueDart  │ S0126     │           17 │    551963.0 │  47.05882352941177 │
│ Kolkata    │ Pune        │ Delhivery │ S0318     │           14 │    544445.0 │ 35.714285714285715 │
│ Delhi      │ Delhi       │ Delhivery │ S0032     │           16 │    53

In [327]:
obt = f"""
select oi.order_item_id, oi.order_id, oi.product_id, oi.seller_id, oi.quantity, c.city as customer_city,
o.customer_id, p.category, p.subcategory, p.base_price,  s.carrier, s.shipped_at, s.delivered_at, 
s.ship_from_city, s.ship_to_city, s.shipping_cost, s.delivery_status from order_items oi
join orders o on oi.order_id = o.order_id
join customers c on o.customer_id = c.customer_id
join products p on oi.product_id = p.product_id
join shipments s on oi.order_id = s.order_id
"""

In [328]:
duckdb.sql(obt)

┌───────────────┬───────────┬────────────┬───────────┬──────────┬───────────────┬─────────────┬────────────────┬────────────────┬────────────┬───────────┬─────────────────────┬─────────────────────┬────────────────┬──────────────┬───────────────┬─────────────────┐
│ order_item_id │ order_id  │ product_id │ seller_id │ quantity │ customer_city │ customer_id │    category    │  subcategory   │ base_price │  carrier  │     shipped_at      │    delivered_at     │ ship_from_city │ ship_to_city │ shipping_cost │ delivery_status │
│    varchar    │  varchar  │  varchar   │  varchar  │  int64   │    varchar    │   varchar   │    varchar     │    varchar     │   double   │  varchar  │       varchar       │       varchar       │    varchar     │   varchar    │    double     │     varchar     │
├───────────────┼───────────┼────────────┼───────────┼──────────┼───────────────┼─────────────┼────────────────┼────────────────┼────────────┼───────────┼─────────────────────┼─────────────────────┼───────

In [333]:
# 1. Run the query and convert to a dataframe
df = duckdb.sql(obt).df()

# 2. Export the dataframe to a CSV
df.to_csv('my_analysis_results.csv', index=False)